In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv


C:\Users\abhij\AppData\Local\Temp\ipykernel_21352\1576698513.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\LangGraph\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
loader = PyPDFLoader("Valorant_RAG_Handbook.pdf")

document = loader.load()

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

chunks = splitter.split_documents(document)

print(len(chunks))

169


In [12]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9351.75it/s]


In [7]:
vector_store = FAISS.from_documents(chunks, embeddings)

In [8]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [19]:
query = "Who is phoenix"
result = retriever.invoke(query)
context = [doc.page_content for doc in result]
metadata = [doc.metadata for doc in result]

prompt = f"Answer the following query: \n {query}, \n using the following context {context} \n Metadata: {metadata}"

In [ ]:
response = llm.invoke(prompt)
if isinstance(response, str):
    print(response)
elif isinstance(response.content, list):
    if isinstance(response.content[0], str):
        print(response.content[0])
    elif isinstance(response.content[0], dict):
        print(response.content[0].get("text", ""))
else:
    print("")

Based on the provided context, Phoenix is a character who uses fire-based abilities and has the ability to heal himself.
